In [2]:

import pandas as pd
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

print("All libraries imported successfully!")

All libraries imported successfully!


# PDF QUESTION ANSWERING SYSTEM USING RAG

## Project Overview

This project uses Retrieval Augmented Generation (RAG) to answer questions from PDF documents.

### Technologies Used
- Python
- LangChain
- ChromaDB
- Sentence Transformers
- Ollama
- Pandas 

In [3]:
from pathlib import Path

# Set project directory
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "dataset").exists():
    BASE_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "dataset").exists():
    BASE_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError("Dataset folder not found!")

# Set folder paths
DATASET_DIR = BASE_DIR / "dataset"
CHROMA_DIR = BASE_DIR / "chroma_db"

print("Project paths configured successfully!")
print("Project Directory:", BASE_DIR)
print("Dataset Directory:", DATASET_DIR)
print("ChromaDB Directory:", CHROMA_DIR)

Project paths configured successfully!
Project Directory: d:\PDF_Question_Answering\PDF_Question_Answering
Dataset Directory: d:\PDF_Question_Answering\PDF_Question_Answering\dataset
ChromaDB Directory: d:\PDF_Question_Answering\PDF_Question_Answering\chroma_db


In [4]:
# Load document chunks

chunks_file = DATASET_DIR / "chunks.csv"

df = pd.read_csv(chunks_file)

chunks = df["Text"].dropna().astype(str).tolist()

print("Document Chunks Loaded Successfully!")
print("Total Chunks:", len(chunks))

# Display first 3 chunks
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}:")
    print(chunk[:300])

Document Chunks Loaded Successfully!
Total Chunks: 184

Chunk 1:
9
UNIT 1  INTRODUCTION TO ARTIFICIAL     
      INTELLIGENCE
Structure
1.1  Introduction
1.2  Objectives
1.3  Basics of Artificial Intelligence (AI)?
1.4  Brief history of Artificial Intelligence 
1.5  Components of Intelligence
1.6  Approaches to Artificial Intelligence
1.7  Comparison between Arti

Chunk 2:
1.9.1  Stimulus - Response Agents
1.10 Summary
1.11 Solutions/Answers
1.12 Further Readings 
1.1 INTRODUCTION 
Today, artificial intelligence is used in a wide variety of applications, including 
engineering, technology, the military, opinion mining, sentiment analysis, 
and  many more. It is also u

Chunk 3:
to its presence. It is utilised in systems that recognise both voices and faces. 
In addition to this, it can provide you with shopping recommendations that 
are tailored to your own purchasing preferences. Finding spam and preventing 
fraudulent use of credit cards is made much easier when you have


In [5]:
print("Loading Embedding Model...")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully!")

Loading Embedding Model...


d:\PDF_Question_Answering\PDF_Question_Answering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1603.49it/s]


Embedding Model Loaded Successfully!


In [6]:
print("Connecting to ChromaDB...")

vector_db = Chroma(
    collection_name="pdf_documents",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR)
)

print("ChromaDB Connected Successfully!")

total_documents = vector_db._collection.count()

print("Total Documents Stored:", total_documents)

Connecting to ChromaDB...
ChromaDB Connected Successfully!
Total Documents Stored: 184


In [7]:
print("Loading Ollama LLM...")

llm = ChatOllama(
    model="gemma3:1b",
    temperature=0
)

print("Ollama Model Ready!")

Loading Ollama LLM...
Ollama Model Ready!


In [8]:
def answer_question(question):

    print("\nSearching relevant PDF content...")

    # Retrieve top 5 relevant chunks
    docs = vector_db.similarity_search(
        question,
        k=5
    )

    # Combine retrieved content
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    print("Relevant Documents Retrieved:", len(docs))

    # Create prompt
    prompt = f"""
    Answer the question using only the following context.

    Context:
    {context}

    Question:
    {question}

    Provide a clear and simple answer.
    If the answer is not available in the context,
    say that the information is not available.
    """

    # Generate answer
    response = llm.invoke(prompt)

    return response.content

In [9]:
question = "What is artificial intelligence?"

answer = answer_question(question)

print("\n================================")
print("PDF QUESTION ANSWERING RESULT")
print("================================")

print("\nQuestion:", question)

print("\nAnswer:")
print(answer)


Searching relevant PDF content...
Relevant Documents Retrieved: 5

PDF QUESTION ANSWERING RESULT

Question: What is artificial intelligence?

Answer:
Artificial intelligence is the ability of a machine to display human-like capabilities such as reasoning, learning, planning, and creativity. It’s a system built to do a specific job.


In [10]:
questions = [
    "What is artificial intelligence?",
    "What are the applications of artificial intelligence?",
    "Explain machine learning.",
    "What is deep learning?",
    "How does AI work?",
    "What are the benefits of artificial intelligence?",
    "What are the challenges of AI?",
    "Explain neural networks.",
    "What is natural language processing?",
    "How is AI used in healthcare?",
    "What is computer vision?",
    "What is supervised learning?",
    "What is unsupervised learning?",
    "What is reinforcement learning?",
    "What is the future of artificial intelligence?"
]

print("Questions Prepared Successfully!")
print("Total Questions:", len(questions))

Questions Prepared Successfully!
Total Questions: 15


In [11]:
results = []

for i, question in enumerate(questions, start=1):

    print(f"\nProcessing Question {i}: {question}")

    answer = answer_question(question)

    results.append({
        "Question": question,
        "Answer": answer
    })

print("\nAll Questions Processed Successfully!")


Processing Question 1: What is artificial intelligence?

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing Question 2: What are the applications of artificial intelligence?

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing Question 3: Explain machine learning.

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing Question 4: What is deep learning?

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing Question 5: How does AI work?

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing Question 6: What are the benefits of artificial intelligence?

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing Question 7: What are the challenges of AI?

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing Question 8: Explain neural networks.

Searching relevant PDF content...
Relevant Documents Retrieved: 5

Processing 

In [12]:
answers_df = pd.DataFrame(results)

output_file = DATASET_DIR / "answers.csv"

answers_df.to_csv(output_file, index=False)

print("Answers Saved Successfully!")
print("Total Answers:", len(answers_df))
print("Saved Location:", output_file)

Answers Saved Successfully!
Total Answers: 15
Saved Location: d:\PDF_Question_Answering\PDF_Question_Answering\dataset\answers.csv


In [13]:
# Display generated answers

print("PDF QUESTION ANSWERING - RESULTS")
print("=================================")

print("Total Questions:", len(answers_df))

print("\nGenerated Answers:")

display(answers_df)

PDF QUESTION ANSWERING - RESULTS
Total Questions: 15

Generated Answers:


,Question,Answer
0,What is artificial intelligence?,Artificial intelligence is the ability of a ma...
1,What are the applications of artificial intell...,"According to the context, artificial intellige..."
2,Explain machine learning.,Machine learning is a branch of artificial int...
3,What is deep learning?,Deep learning is a subfield of machine learnin...
4,How does AI work?,AI is the ability of a machine to display huma...
5,What are the benefits of artificial intelligence?,The context doesn’t provide any information ab...
6,What are the challenges of AI?,The context doesn’t explicitly state what the ...
7,Explain neural networks.,Neural networks are made up of layers of softw...
8,What is natural language processing?,Natural language processing (NLP) is the study...
9,How is AI used in healthcare?,AI is used in healthcare to address issues of ...


In [14]:
# Calculate answer statistics

answers_df["Answer_Length"] = answers_df["Answer"].apply(len)

print("PROJECT EVALUATION")
print("==================")

print("Total Questions:", len(answers_df))
print("Total Answers:", answers_df["Answer"].notna().sum())

print("Average Answer Length:",
      round(answers_df["Answer_Length"].mean(), 2))

print("Empty Answers:",
      answers_df["Answer"].isna().sum())

print("\nEvaluation Completed Successfully!")

PROJECT EVALUATION
Total Questions: 15
Total Answers: 15
Average Answer Length: 217.07
Empty Answers: 0

Evaluation Completed Successfully!


In [15]:
evaluation_file = DATASET_DIR / "evaluation_results.csv"

answers_df.to_csv(evaluation_file, index=False)

print("Evaluation Results Saved!")
print(evaluation_file)

Evaluation Results Saved!
d:\PDF_Question_Answering\PDF_Question_Answering\dataset\evaluation_results.csv
